# 2 — What do simple models recover from the cue?

This notebook is a research guide to the **PCA and LDA reference experiment**.
Its question is whether ordered endpoint pair, start and goal can be recovered
from a source-separated, condition-matched cue pseudopopulation, and how sensitive
that recovery is to the real training-source pool. It builds from fresh inputs;
Notebook A is useful background but is not an execution dependency.

Read each decision before its code, then use the resulting diagnostics to decide
what can be defended. We first justify observations and source separation, then
trace normalization and feature shapes, compare raw/PCA logistic and raw LDA
readouts, inspect 3D geometry and source sensitivity, and finish with an artifact
replay. The [research walkthrough](00_research_guide.ipynb) connects this bounded
experiment to the historical path, endpoint and joint-training work.

**Evidence status:** the default invented inputs are a software smoke test, with
three landmarks and six ordered pairs; their scores are not neuroscience evidence.
The supplied biological route is a separately gated exploratory analysis with six
landmarks/30 pairs and a selected complete-support panel. Neither is confirmatory.
The current notebook is unexecuted teaching source with empty outputs; saved older
execution records do not validate these revised cells. Nothing here changes the
accepted fit grid or authorizes another biological run.

In [ ]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
from neuro.notebook_inputs import configuration_from_environment, inspect_inputs, load_inputs, effective_pair_map
from neuro.pseudopopulation import sample_plan, materialize, support
from neuro.notebook_artifacts import ArtifactStore, FitAccounting, target_heads, write_private_json, verify_artifacts, runtime_identity, validate_kernel_identity
from neuro.notebook_support import workspace, usage_summary, source_budget, checked_assemblies, compact_metrics, plot_confusions

configuration = configuration_from_environment()
settings = configuration['execution']
output = workspace()
def private_warning(message, category, filename, lineno, file=None, line=None):
    # Preserve full provenance privately; do not render host paths in shared figures/output.
    warning_path = output / 'warnings-private.jsonl'
    with warning_path.open('a') as handle:
        handle.write(json.dumps({'message': str(message), 'category': category.__name__, 'filename': filename, 'lineno': lineno}) + '\n')
    warning_path.chmod(0o600)
    print(category.__name__ + ': ' + str(message), file=sys.stderr)
warnings.showwarning = private_warning
store = ArtifactStore(output / 'artifacts')
store.save('configuration', configuration)
store.save('kernel_runtime_before_inputs', validate_kernel_identity(configuration))
print('Input mode:', configuration['mode'], '| exploratory computational assay; no biological IID inference')

## Establish what the observations and targets mean

The direct measurements are sorted spike timestamps and recorded task-event markers.
A sorted unit is an inferred spike cluster, not a census of every hippocampal neuron.
For lab inputs, the analysis converts spikes into counts in ten half-open 100-ms
cue bins spanning 0–1 s from cue onset; the smaller invented fixture has four bins.
The marker-derived start and goal are provisional task labels:
agreement between recording streams is a consistency check, not independent proof of
the experiment's semantic meaning. Source eligibility, exclusions and the marker
join must be inspected before interpreting a decoder.

One source trial records only the units in its session block. A zero means an observed
bin containing no spikes; an off-block NaN means that unit was not recorded in that
source. Treating both as zero would manufacture measurements. To form a complete
pseudopopulation row, we match one source from **each block on the known ordered
start→goal pair** and copy that block's units and all bins together. Each resulting
row is assembled, not simultaneously recorded. This preserves within-block observations
while imposing a cross-block combination; it cannot preserve unmeasured cross-session
noise correlations or reveal a real animal's full population state.

This construction is defensible for a **condition-level assay** when every retained
block supports every requested condition in each split, units and timing agree, and
source contributors stay separated. It is not an unknown-label single-trial prediction
pipeline: the condition is needed before constructing the input. Balanced numbers of
assembled rows do not imply balanced numbers of real sources. Sparse block/pair pools
are reused more, some source measurements are never drawn, and complete-support
selection changes the population being studied. Notebook A exposes the selection
waterfall, support matrices, exclusions and source reuse in greater detail; this
notebook independently reconstructs its own inputs and repeats the core gates.

For supplied lab inputs, use only the original outer-training partition. Keep the
historically exposed development assignment frozen, embargo any training example
sharing a current or predecessor contributor with **any original development
example**, then select the complete-support panel. The predecessor matters because
the previous trial contributes the current start label. The historical source route
selected 27 units in 10 sessions; the 195-unit/58-session compact inventory does not
mean that all those units survive this analysis. The printed audit is authoritative
for this execution. Contributor separation still leaves session and longer-range
serial dependence, and development is not a pristine test set.

**Read the next outputs as a gate.** Inspect selected train/development counts,
exclusion reasons, contributor overlap and the source×unit×bin tensor. Missing support,
unexplained selection differences or overlap require investigation before fitting;
they are not reasons to silently repair labels or fill absent neural values.

In [ ]:
inspection = inspect_inputs(configuration)  # Metadata/hash step only for supplied inputs.
store.save('metadata_selection', inspection)  # Protected provenance; never displayed.
print(json.dumps(inspection.audit, sort_keys=True))
assert inspection.audit['contributor_overlap'] == 0
assert inspection.audit['selected_train'] > 0 and inspection.audit['selected_development'] > 0

In [ ]:
data, neural_audit = load_inputs(inspection)  # Supplied neural path remains explicitly review-gated.
store.save('source_data', data)
store.save('neural_audit', neural_audit)
print(json.dumps(neural_audit, sort_keys=True))
assert np.isfinite(data.values[data.observed]).all()
assert np.isnan(data.values[~data.observed]).all()
assert not data.dependency_audit()['overlapping_contributor_ids']
print('Source × unit × bin shape:', data.values.shape, '| units:', data.value_unit, '| clock units:', data.time_unit)

## Freeze the comparison before learning from spikes

We hold the development **values, labels, exposure and contributing sources** fixed
across observed seeds, representations and source budgets. The function
`checked_assemblies` tests this equality, not merely an equal row count. Each observed
training seed changes the constructed training sample (and CEBRA initialization and
sampler when present), while development assembly seed remains zero. These are
computational perturbations, not new subjects or independent biological replications.

The full/reduced intervention asks how sensitive this fixed workflow is to its
available training-source pool. Within every block×condition pool, reduced mode keeps
a deterministic nested `floor(n/2)` subset using the same fixed subset seed. The unit
panel, condition list, bin definitions and synthetic-row budget stay fixed. Every
learned transformation and decoder is refitted on the new training assembly. This
is not deletion of half the neurons, a new development set, or an exact 50% reduction
in independent measurements: floor rounding and source reuse matter. Inspect eligible,
actually used, unused and repeated counts alongside the score changes.

The independent provenance unit is a real source trial **plus its target
contributors**, with trials nested in sessions. Our estimand is descriptive
recoverability of the matched condition in this selected pseudopopulation, under the
declared source split and construction. A higher development score supports a better
readout of these constructed examples. It does not establish trial-specific path
prediction, intention, causal mechanism or generalization to a new animal/session.

In [ ]:
from neuro.pca_lda import fit_standardizer, raw_features, fit_pca, fit_lda
from neuro.notebook_support import fit_logistic
observed = effective_pair_map(data, permutation_seed=None)
dev_plan = sample_plan(data, split_id='development', rows_per_condition=settings['development_rows_per_condition'], seed=0, effective_conditions=observed)
reference_development = materialize(data, dev_plan)
store.save('fixed_development', reference_development)
accounting = FitAccounting().start()
results = []
from neuro.notebook_diagnostics import metrics_table, plot_decoder_summary, embedding_summary, plot_embedding_views
case_diagnostics = {}
reference_view = None

## Follow the feature tensor: counts → standardized vectors → readout

Let X have shape **R assembled rows × U units × T cue bins**. Encoder normalization
pools only the R×T **training** unit vectors, estimating one mean and scale for each
unit. It then applies those frozen values to development. Pooling improves the
numerical definition of each feature's scale; it does not make adjacent bins independent.
The values fed to the models are standardized counts, not firing rates or directly
measured latent states. A nearly silent unit and a missing unit are different cases;
the source contract must already have resolved observation support.

The raw baseline flattens X as unit-major features: unit 1 at bins 1…T, then unit 2,
and so on. PCA fits a full-SVD linear projection to the R×T training vectors after
the same kind of train-only unit standardization. Its eight components maximize
retained training variance, without start/goal labels. It requires numerical training
rank ≥8 and does not pad a deficient representation. PCA output is flattened
**time-major**: the eight coordinates at bin 1, then eight at bin 2. Both retain time
order, but column indices have different meanings. The shown dimensions below make
the distinction checkable. Explained-variance ratios quantify retained standardized
training variance; they do not measure task information or decoder performance.

Each logistic readout additionally fits its own training-only per-feature scaler.
This step has U×T columns for raw features and 8×T for PCA, and differs from the
earlier unit scaler shared across time. A separate classifier is fitted for each
target. LDA here is a **supervised SVD classifier on the raw whole-cue features**,
with its own scaler. Its class-conditional, shared-covariance linear decision rule
is another readout assumption; it is not an unsupervised eight-dimensional embedding
and no LDA scatterplot is used to claim label-free separation. These choices are
visible in [the local implementation](../src/neuro/pca_lda.py) and the
[official LDA reference](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html).

The grid below has two observed seeds × two source budgets. It fits four PCA models,
24 logistic heads and 12 LDA classifiers. It records each bundle and fitted state so
the final cell can replay transformations and predictions with fitting blocked.
That replay checks computational reproducibility; it cannot validate the biological
meaning of the labels, the appropriateness of source selection or the scientific claim.

## Decide what a successful decoder would mean

The decoder sees the entire cue, with bins concatenated in order; it does not decode
each time bin separately. We fit **three independent classification heads**: ordered
pair, start, and goal. Pair classification asks which joint endpoint condition was
used to assemble a row; start/goal ask its components. None asks which ordered path
the animal took beyond those endpoints. A strong goal head can coexist with a weak
pair head, and an easy marginal target must not be presented as successful path decoding.

For K supported classes, balanced accuracy is the mean of the K class recalls.
Accuracy counts all correct rows equally; macro-F1 averages each class's precision/recall
tradeoff. With truly equal evaluation support, accuracy and balanced accuracy coincide;
we still show both to expose support mistakes and make the metric definition explicit.
Log loss is the average negative log probability assigned to the correct class, so
it can worsen when a model becomes confidently wrong even if hard accuracy improves.
Uniform-probability references are **1/K balanced accuracy** and **ln(K) log loss**.
They are descriptive references, not significance cutoffs; macro-F1 has no general
1/K reference. See the [official metric definitions](https://scikit-learn.org/stable/modules/model_evaluation.html).

Always read count **and row-normalized** confusion matrices. Rows are true classes,
columns predicted classes; the diagonal of the normalized matrix is per-class recall.
A concentrated prediction column reveals a decoder favoring one label, even if a
single average looks acceptable. The matrices describe constructed rows, not independent
source-trial counts. The training-minus-development balanced-accuracy gap is a
diagnostic of fitting versus transfer within this assay: a large positive gap invites
an overfitting/distribution-shift explanation; a small gap with low scores is not success.

Compare representations on the same development rows, head, seed and source budget.
Compare raw logistic with PCA/CEBRA logistic to assess the consequences of the
representation under this readout choice; compare raw LDA with raw logistic to assess
the classifier assumption. Fixed regularization is a controlled workflow choice, not
proof that every representation has its optimal decoder. Do not choose the best seed
or hyperparameter after looking at development and call that a held-out result.
The figures below show individual cases and references; no confidence interval or
biological p-value is inferred from pseudorows or the two seeds.

In [ ]:
for seed in settings['seeds']:
    for budget in ('full', 'reduced'):
        selected, effective = source_budget(data, observed, reduced=budget == 'reduced', subset_seed=settings['subset_seed'])
        train, development = checked_assemblies(selected, effective, dev_plan, reference_development, config=settings, seed=seed)
        key = f'observed_seed{seed}_{budget}'
        store.save_bundle(key, train, development)
        print(key, 'source accounting:', json.dumps(usage_summary(selected, train), sort_keys=True))
        standardizer = fit_standardizer(train.values)
        raw_train, raw_dev = raw_features(standardizer.transform(train.values)), raw_features(standardizer.transform(development.values))
        pca = fit_pca(train.values, n_components=settings['pca_components'])
        pca_train, pca_dev = pca.transform(train.values), pca.transform(development.values)
        store.save_encoder(key + '_raw', standardizer, kind='raw')
        store.save_encoder(key + '_pca', pca, kind='pca')
        print(key, 'PCA explained variance:', pca.pca.explained_variance_ratio_.round(4).tolist())
        print(key, 'Tensor/feature shapes:', {'row_unit_bin': train.values.shape,
              'raw_unit_major': raw_train.shape, 'pca_time_major': pca_train.shape})
        geometry = embedding_summary(pca_train, pca_dev, n_bins=train.values.shape[2])
        case_diagnostics[key] = {'pca_geometry': geometry}
        print(key, 'PCA geometry:', json.dumps(geometry, sort_keys=True))
        if key == 'observed_seed0_full':
            reference_view = {'key': key, 'train': pca_train.copy(), 'development': pca_dev.copy(),
                              'targets': target_heads(development), 'n_bins': train.values.shape[2],
                              'unit_mean': standardizer.scaler.mean_.copy(),
                              'unit_scale': standardizer.scaler.scale_.copy(),
                              'variance_ratio': pca.pca.explained_variance_ratio_.copy()}
        run_results = []
        for representation, train_x, dev_x in (('raw', raw_train, raw_dev), ('pca', pca_train, pca_dev)):
            for head, y in target_heads(train).items():
                model = fit_logistic(train_x, y, C=settings['logistic_C'], max_iter=settings['logistic_max_iter'])
                row = store.readout(key + '_' + representation + '_' + head, model, encoder=key + '_' + representation,
                                    bundle=key, head=head, train_features=train_x, development_features=dev_x)
                run_results.append(row)
        for head, y in target_heads(train).items():
            model = fit_lda(raw_train, y)
            row = store.readout(key + '_lda_' + head, model, encoder=key + '_raw', bundle=key, head=head,
                                train_features=raw_train, development_features=raw_dev)
            run_results.append(row)
        results.extend(run_results)
        print(json.dumps(compact_metrics(run_results), sort_keys=True))
        fig = plot_confusions(run_results, title=key + ': development counts and row-normalized confusions')
        display(fig); plt.close(fig)
store.save('aggregate_metrics', results)
store.save('diagnostic_summaries', case_diagnostics)

## Read the scores before looking at the clouds

Each table row names its source budget, seed, representation and target. The summary
shows train/development performance together; a seed is a computational sensitivity
case, not an error bar. The text below reports the numbers actually produced by this
execution rather than assuming that PCA or LDA must win. Positive full-minus-reduced
differences indicate sensitivity to source availability under this one nested subset;
they do not establish a sample-efficiency law or a causal biological effect.

In [ ]:
# Summaries use already-computed predictions; no model is fitted in this cell.
from html import escape
from IPython.display import HTML
metric_rows = metrics_table(results)
columns = [('world', 'World'), ('seed', 'Seed'), ('budget', 'Sources'), ('model', 'Model'),
           ('head', 'Target'), ('classes', 'K'), ('train_BA_percent', 'Train BA %'),
           ('development_BA_percent', 'Development BA %'), ('uniform_BA_percent', 'Uniform BA %'),
           ('gap_percentage_points', 'Gap (pp)'), ('accuracy_percent', 'Accuracy %'),
           ('macro_F1', 'Macro F1'), ('log_loss_nats', 'Log loss (nats)'),
           ('uniform_log_loss_nats', 'Uniform loss (nats)')]
def metric_text(value):
    return f'{value:.3f}' if isinstance(value, (float, np.floating)) else str(value)
header = '<tr>' + ''.join('<th>' + escape(label) + '</th>' for _, label in columns) + '</tr>'
body = ''.join('<tr>' + ''.join('<td>' + escape(metric_text(row[key])) + '</td>'
                              for key, _ in columns) + '</tr>' for row in metric_rows)
display(HTML('<div style="overflow-x:auto"><table><caption>Matched whole-cue readouts; '
             'constructed rows, no biological uncertainty intervals</caption><thead>'
             + header + '</thead><tbody>' + body + '</tbody></table></div>'))
fig = plot_decoder_summary(results)
display(fig); plt.close(fig)
print('How to read this execution:')
for row in results:
    score = row['development']
    k = len(score['classes'])
    ba = score['balanced_accuracy']
    gap = row['balanced_accuracy_gap']
    relation = 'above' if ba > 1 / k else ('below' if ba < 1 / k else 'at')
    print(f"{row['name']}: development BA {ba:.3f} is {relation} uniform {1 / k:.3f}; "
          f"train minus development {gap:+.3f}; log loss {score['log_loss']:.3f} vs uniform {np.log(k):.3f}.")
print('These are constructed-row summaries, without biological significance or independence claims.')
by_name = {row['name']: row for row in results}
example = by_name['observed_seed0_full_raw_pair']['development']
counts = np.asarray(example['confusion_counts'], dtype=float)
class_support = counts.sum(axis=1)
assert np.all(class_support > 0)
recall = np.diag(counts) / class_support
worked_ba = recall.mean()
worked_accuracy = np.trace(counts) / counts.sum()
np.testing.assert_allclose(worked_ba, example['balanced_accuracy'])
np.testing.assert_allclose(worked_accuracy, example['accuracy'])
print('Worked confusion arithmetic: observed seed 0 / full / raw logistic / pair')
print('True-class supports:', class_support.astype(int).tolist())
print('Diagonal / true-class support = recalls:', np.round(recall, 3).tolist())
print(f'Mean class recall = BA {worked_ba:.3f}; diagonal sum / all rows = accuracy {worked_accuracy:.3f}.')
worst = int(np.argmin(recall))
print('Lowest recall class (first on ties):', example['classes'][worst], f'{recall[worst]:.3f}')
print('Matched source-budget sensitivity (full minus reduced development BA):')
for name, full in by_name.items():
    if '_full_' in name:
        reduced = by_name[name.replace('_full_', '_reduced_', 1)]
        delta = full['development']['balanced_accuracy'] - reduced['development']['balanced_accuracy']
        print(f"{name}: {delta:+.3f}; same development rows within this world/seed.")
print('Inspect the source reuse reports and confusion rows before attributing a difference to representation quality.')

## Inspect the distribution in three dimensions

The following display is fixed in advance to **observed seed 0, full sources**. It
uses the PCA learned on that case's training data, applied unchanged to its matched
development rows. It does not accidentally use whichever reduced case happened to
finish last. The three-dimensional panels show the **first three existing PCA
coordinates at the first cue bin**, colored in turn by start and goal, with gray
training points behind them. Pair information is assessed by its readout. No
second dimensionality-reduction model is fitted for the picture. The companion
all-bin view uses those same coordinates colored by native bin to reveal temporal
organization; bins and repeated source combinations are dependent observations.

Each plotted first-bin point is one condition-matched synthetic population row.
Distances are in PCA coordinates of standardized counts, not physical positions in
the maze. The model and whole-cue readout can use eight coordinates at every bin,
so a three-axis first-bin projection can hide information that the decoder retains.
Conversely, visible clusters may reflect assembly, endpoint matching, time or a
small group of heavily reused sources. The development geometry and train/development
variance/effective-rank summaries are useful diagnostics, not evidence of a neural manifold's
topology or an independent population trajectory.

Before interpreting separation, connect it to the exact head's development score,
confusion pattern and full/reduced sensitivity. Ask whether the result survives
removing source support, whether minority pools dominate reuse, and whether a raw
readout performs just as well. The selected 3D illustration is a reading aid;
the per-case numerical diagnostics above cover all four fitted PCA cases.

In [ ]:
assert reference_view is not None and reference_view['key'] == 'observed_seed0_full'
print('Displayed case:', reference_view['key'])
print('Frozen training unit means/scales:', reference_view['unit_mean'].round(3).tolist(), reference_view['unit_scale'].round(3).tolist())
print('Training explained-variance ratio:', reference_view['variance_ratio'].round(4).tolist())
fig = plot_embedding_views(reference_view['train'], reference_view['development'], reference_view['targets'],
                           n_bins=reference_view['n_bins'], title='Observed seed 0 / full sources / PCA')
display(fig); plt.close(fig)
print('Per-case diagnostics cover:', sorted(case_diagnostics))

## Interpretation limits
These are source-safe conditional pseudopopulations, not simultaneous trials. Matching uses known labels; this is not an unknown-trial prediction pipeline. Bins, synthetic rows and computational seeds are dependent. Place-selected units, provisional marker meaning, inherited preprocessing and historical exposure remain limitations. Contributor separation does not remove long-range serial/session confounds. No biological confidence intervals, p-values, mechanism, dynamics or prospective-path claim follows from these figures. The permutation diagnostic is alignment-breaking, not a biological exchangeability test.

## What the next researcher should be able to defend

**Source trial** means a recorded trial, with a predecessor contributor when required
for label derivation. **Block** is the group of units copied from one recording
session. **Pseudorow** is the condition-matched assembly across blocks; its count is
not biological n. **Representation/encoder** maps neural vectors to features;
**readout/decoder** predicts an operational target from those features. **Development**
means the fixed, historically exposed evaluation assignment here, not an untouched
test set. **Observability in this notebook** means that decisions, intermediate
quantities, failures and provenance can be inspected. **Formal state observability**
would require a specified dynamical system and measurement model showing which
hidden states can be distinguished; classification accuracy does not provide that.

Before handing off a result, write an evidence paragraph containing its target,
source cohort, synthetic counts, split/embargo, time window, representation/readout,
score relative to a named reference, seed and source-budget sensitivity, and strongest
allowed claim. A useful conclusion can be negative: “the encoder's objective improved,
but the matched decoder did not improve over the raw reference” is a complete research
observation. Avoid changing “not established” into evidence that hippocampal coding
is absent. Historical H1 concerns path information beyond endpoints and reproducible
organization; these endpoint heads do not settle it.

As a reading exercise, trace one tensor column back to unit and cue bin, then trace
one assembled row back to its block contributors without exporting their identities.
Explain why class balance in that row table can coexist with skewed source reuse.
Choose one head and reconcile its mean score with the worst recall in its confusion
matrix. Explain why the full/reduced difference is a source-sensitivity result and
why a change in a three-axis display is not automatically a readout improvement.
Finally, list which claims remain impossible even if every software assertion passes.
These exercises require inspecting existing outputs, not running additional models.

The final cell closes fit accounting, authenticates this execution's artifact manifest
and replays its saved transformations/predictions with fitting blocked. Read its
status as a computational check. A defensible scientific result additionally needs
the source/label reasoning above and an independent review of leakage, dependence,
confounds and the written claim. This source notebook's outputs are intentionally
cleared; the revised figures are specifications until an authorized execution produces
them. Earlier successful runs attest their recorded code, not these newly added cells.

In [ ]:
fit_counts = accounting.stop()
store.save('kernel_runtime_final', runtime_identity())
expected_counts = {'pca': 4, 'logistic': 24, 'lda': 12, 'cebra': 0, 'optimizer_updates': 0}
manifest_hash = store.finish(fit_counts=fit_counts, expected_counts=expected_counts)
replay = verify_artifacts(output / 'artifacts', expected_manifest_hash=manifest_hash)
write_private_json(output / 'completion.json', {'manifest_sha256': manifest_hash, 'fit_counts': fit_counts, 'replay': replay})
print(json.dumps(replay, sort_keys=True))
assert replay['status'] == 'fit-blocked-loaded-model-replay-passed'